# Comparing Image Embeddings: MobileNetV2 vs. EfficientNetV2M vs. CLIP

Three ways to turn an image into a vector:

| Method | What it is | Training signal | Modality | Typical dim |
|---|---|---|---|---|
| **MobileNetV2** (Keras Applications) | Lightweight, frozen ImageNet CNN (2018) | Supervised classification (1000 ImageNet classes) | Vision only | 1280 |
| **EfficientNetV2M** (Keras Applications) | Larger, frozen ImageNet CNN (2021) | Supervised classification (1000 ImageNet classes) | Vision only | 1280 |
| **CLIP** (via `fastembed`) | Vision-language contrastive transformer, run on ONNX Runtime | Contrastive image-text matching (400M web pairs) | Vision **and** text, same space | 512 |

The first two share a training signal (ImageNet classification) but differ in architecture and
scale -- does a bigger, newer CNN produce a meaningfully different embedding space than a small,
older one? The third method changes the training signal itself: instead of predicting a fixed label
set, CLIP aligns images with their captions, producing a joint vision-language space.

We'll extract all three embeddings for a small cat/dog image set, then run an identical comparison
battery on each: same-vs-diff-class cosine separation, PCA/t-SNE projections, a linear-probe
classification accuracy, and a similarity heatmap. Finally, CLIP's unique trick -- querying images
with a text prompt -- is demonstrated, something neither ImageNet-supervised CNN can do regardless
of size.

## Learning objectives

- Extract a fixed-length embedding from an image using a frozen pretrained model.
- Explain how a model's training signal shapes the embedding space it produces.
- Compare embedding spaces using cosine separation, 2D projections, and a linear probe.
- Explain why a linear probe is a fair "quality number" for a frozen embedding.
- Perform zero-shot text-to-image retrieval in CLIP's shared space, and interpret its low absolute
  cosine values correctly.

## Background

`U2-2_CNN-6_TransferLearning.ipynb` established the move this notebook depends on: load a pretrained
model with `include_top=False`, freeze it, and treat its output as a feature vector. Here that vector
is the *object of study* rather than an input to a classifier.

Two preliminaries are worth stating precisely.

**Cosine similarity** measures the angle between two vectors, ignoring their magnitudes:

$$ \cos(\mathbf{a}, \mathbf{b}) = \frac{\mathbf{a} \cdot \mathbf{b}}
   {\lVert \mathbf{a}\rVert \, \lVert \mathbf{b}\rVert} $$

It is the standard way to compare embeddings, because what an embedding encodes is *direction* — two
images of the same thing should point the same way, however long their vectors happen to be. After
L2 normalization the cosine reduces to a plain dot product, which is why every comparison below
normalizes first.

**Preprocessing is per model, not universal.** Each pretrained network expects its inputs scaled the
way it was trained: MobileNetV2 wants pixels mapped to $[-1, 1]$, EfficientNetV2M has that
normalization baked in as a `Rescaling` layer and expects raw 0–255, and CLIP applies its own
pipeline inside `.embed()`. That is why the code keeps one raw, unscaled array and lets each model
transform it — handing a model the wrong scaling produces quietly bad embeddings rather than an
error.

## This notebook covers

1. Loading the cat/dog image set
2. Method 1 — MobileNetV2 embeddings
3. Method 2 — EfficientNetV2M embeddings
4. Method 3 — CLIP embeddings via `fastembed`
5. The comparison battery, applied identically to all three
6. CLIP's zero-shot text-to-image retrieval
7. Review

**Prerequisites:** `U2-2_CNN-6_TransferLearning.ipynb` for frozen pretrained models and feature
extraction; `U1-1_Embeddings-2_Images.ipynb` for image embeddings generally.

**Dataset:** the cat and dog photographs in `Unit 2 - Image Data/Demos/images/` (`images/cat/`,
`images/dog/`), loaded directly from disk.

**Requirements:** `fastembed` (`pip install fastembed`) for the CLIP embeddings — it runs on ONNX
Runtime and downloads roughly 0.6 GB of model weights on first use.

**References:** https://keras.io/api/applications/ and https://github.com/qdrant/fastembed

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import math

pd.set_option('display.max_columns',100)
pd.set_option('display.max_rows',100)

plt.style.use('dark_background')

import warnings
warnings.filterwarnings('ignore')

# Shared course helpers (msds565_helpers.py lives in the repo root).
# Notebooks sit two folders below the root, so '../..' points back to it.
import sys
sys.path.append('../..')
import msds565_helpers as helpers

## 1. Load the images

### 1.1 Configuration

Everything tunable is gathered in one place so the rest of the notebook reads cleanly. Both Keras
models take 224×224 input. The two CLIP entries are separate ONNX models — one per tower — that
nonetheless embed into the same 512-dimensional space, which is what makes the text-to-image
retrieval in section 6 possible.

In [ ]:
# ── CONFIGURATION ─────────────────────────────────────────────
DATA_DIR      = "images/"          # images/cat/, images/dog/
TARGET_SIZE   = (224, 224)         # input size for both Keras models
RANDOM_STATE  = 42

MOBILENET_MODEL_NAME    = "MobileNetV2 (ImageNet, avg-pooled)"
EFFICIENTNET_MODEL_NAME = "EfficientNetV2M (ImageNet, avg-pooled)"
# CLIP runs through fastembed (ONNX Runtime), so each tower is a separate model.
# Both are ONNX ports of sentence-transformers/clip-ViT-B-32 -- OpenAI's CLIP
# ViT-B/32, the same weights this notebook used via transformers -- so they share
# one 512-d space and text-vs-image cosines are meaningful.
CLIP_VISION_MODEL_NAME  = "Qdrant/clip-ViT-B-32-vision"   # 512-d, ~0.34 GB
CLIP_TEXT_MODEL_NAME    = "Qdrant/clip-ViT-B-32-text"     # 512-d, ~0.25 GB

np.random.seed(RANDOM_STATE)

### 1.2 Load and index the dataset

Each subdirectory of `images/` is one class. We keep the images in two forms, and the distinction
matters:

- **`images`** — the original PIL objects at their native sizes. CLIP's `fastembed` wants these,
  because it applies its own resize and crop internally.
- **`X_224`** — a resized, **raw 0–255** float array for the Keras models. It is deliberately left
  unscaled, so each model can apply its own `preprocess_input`.

Keeping one unscaled array and transforming per model is the pattern to take away here. Preprocessing
applied twice, or not at all, is one of the most common causes of silently poor embeddings.

In [ ]:
from pathlib import Path
from PIL import Image


def load_image_dataset(dataset_dir):
    """
    Load all images from a flat two-level dataset directory
    (dataset_dir/<class>/<class>_NNN.jpg). Each subdirectory is a class.
    Reused from BingImageScrape.ipynb.
    """
    dataset_dir = Path(dataset_dir).resolve()
    class_dirs = sorted(p for p in dataset_dir.iterdir() if p.is_dir())

    images, labels = [], []
    for class_dir in class_dirs:
        image_files = sorted(
            p for p in class_dir.iterdir()
            if p.is_file() and p.suffix.lower() in {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
        )
        for img_path in image_files:
            img = Image.open(img_path).convert("RGB")
            img.load()
            images.append(img)
            labels.append(class_dir.name)
        print(f"  {class_dir.name!r}: loaded {len(image_files)} images")

    print(f"\nTotal: {len(images)} images across {len(class_dirs)} classes")
    return images, labels


images, labels = load_image_dataset(DATA_DIR)

label_map = {name: idx for idx, name in enumerate(sorted(set(labels)))}
y = np.array([label_map[l] for l in labels], dtype=np.int64)
print("Classes:", label_map)

# Resized RAW 0-255 float array. Each Keras model applies its OWN preprocess_input
# at inference time (MobileNetV2 rescales to [-1,1]; EfficientNetV2M's normalization
# is baked into the model itself and expects raw 0-255 pixels), so we keep this
# array unscaled here.
X_224 = np.stack([
    np.asarray(img.resize(TARGET_SIZE, Image.LANCZOS), dtype=np.float32)
    for img in images
])
print("X_224 shape:", X_224.shape)

In [ ]:
rng = np.random.default_rng(RANDOM_STATE)
sample_idx = rng.choice(len(images), size=8, replace=False)

fig, axes = plt.subplots(2, 4, figsize=(14, 7))
for ax, i in zip(axes.ravel(), sample_idx):
    ax.imshow(images[i])
    ax.set_title(labels[i])
    ax.axis('off')
plt.tight_layout()
plt.show()

## 2. Method 1 — MobileNetV2

Keras applications: https://keras.io/api/applications/

A lightweight CNN (2018) pretrained for **ImageNet classification**, used here purely as a frozen
feature extractor. `pooling='avg'` global-average-pools the final convolutional feature map to a flat
1280-d vector per image — no fine-tuning. `mobilenet_v2.preprocess_input` rescales raw 0–255 pixels
to $[-1, 1]$.

In [ ]:
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input as mobilenet_preprocess

print(f"Loading {MOBILENET_MODEL_NAME} ...")
mobilenet_model = MobileNetV2(weights='imagenet', include_top=False, pooling='avg', input_shape=(224, 224, 3))
mobilenet_model.trainable = False

Z_mobilenet = mobilenet_model.predict(mobilenet_preprocess(X_224.copy()), batch_size=16, verbose=1)
print("Z_mobilenet shape:", Z_mobilenet.shape)

## 3. Method 2 — EfficientNetV2M

A larger, more recent CNN (2021), also pretrained on ImageNet, used identically as a frozen feature
extractor. Same training signal as MobileNetV2 — the only difference is architecture and scale, which
is exactly the variable we want to isolate.

`efficientnet_v2.preprocess_input` is effectively a no-op here: the normalization is baked into the
model itself as a `Rescaling` layer, so it expects raw 0–255 pixels directly.

In [ ]:
from tensorflow.keras.applications import EfficientNetV2M
from tensorflow.keras.applications.efficientnet_v2 import preprocess_input as efficientnet_preprocess

print(f"Loading {EFFICIENTNET_MODEL_NAME} ...")
efficientnet_model = EfficientNetV2M(weights='imagenet', include_top=False, pooling='avg', input_shape=(224, 224, 3))
efficientnet_model.trainable = False

Z_efficientnet = efficientnet_model.predict(efficientnet_preprocess(X_224.copy()), batch_size=16, verbose=1)
print("Z_efficientnet shape:", Z_efficientnet.shape)

## 4. Method 3 — CLIP (via `fastembed`)

CLIP (Contrastive Language-Image Pretraining) learns a **shared embedding space for images and text**
by pulling matching image/caption pairs together and pushing mismatched pairs apart. Because text
lives in the same space, we can later query images with a sentence -- something neither
ImageNet-supervised CNN above can do.

We load it with **fastembed**, which runs an ONNX export of that same model on ONNX Runtime. Two
practical reasons to prefer it here over `transformers`:

- **No torch.** fastembed's whole dependency list is ONNX Runtime, `tokenizers`, `pillow` and a few
  small utilities. Nothing in this notebook needs a deep-learning framework besides the Keras one we
  already use, and dropping torch also drops the `KMP_DUPLICATE_LIB_OK` workaround this notebook used
  to need (torch and TensorFlow each ship their own OpenMP runtime, and loading both on Windows can
  crash the kernel).
- **A stable API.** The `transformers` route is version-sensitive: `get_image_features()` returns a
  plain tensor on transformers 4.x but a `BaseModelOutputWithPooling` on 5.x, so the same line either
  works or raises `AttributeError` depending on what got installed. fastembed's `.embed()` just
  returns numpy arrays.

CLIP's two towers ship as two separate fastembed models -- vision here, text in section 6 for the
retrieval demo.

In [ ]:
from fastembed import ImageEmbedding

print(f"Loading {CLIP_VISION_MODEL_NAME} ...")
clip_vision = ImageEmbedding(model_name=CLIP_VISION_MODEL_NAME)

# .embed() takes PIL Images directly -- no processor, no tensors, no no_grad() -- and
# returns a generator, so we materialise it into an array.
#
# We hand it `images` (the original PIL objects), NOT the resized X_224 the Keras models
# use: fastembed applies CLIP's own preprocessing internally (resize -> center-crop ->
# CLIP's mean/std), and feeding it a pre-resized array would apply that twice.
Z_clip = np.array(list(clip_vision.embed(images)))

# fastembed L2-normalizes its output by default; transformers does not. That is the entire
# reason fastembed and HuggingFace CLIP embeddings look different if you compare them raw.
# It is a no-op for us, since every comparison below re-normalizes anyway.
print("Z_clip shape:", Z_clip.shape)
print("already unit-norm:", np.allclose(np.linalg.norm(Z_clip, axis=1), 1.0))

## 5. Comparison battery

The same four diagnostics, applied identically to all three embeddings.

Each asks a different question, and they are ordered from coarse to decisive: *are same-class pairs
closer than different-class pairs?* (5.1), *does the structure survive a squeeze to two dimensions?*
(5.2), *how well does a linear model separate the classes?* (5.3), and *what do individual pairs of
images actually score?* (5.4).

### 5.1 Same-vs-diff-class cosine similarity

The most direct test of whether an embedding is any good: sample thousands of random image pairs,
compute the cosine between each, and plot the same-class and different-class distributions
separately.

A useful embedding pushes those two histograms apart. Overlap is error — pairs that the space cannot
distinguish. Note that *where* the distributions sit matters much less than how far apart they are;
different models produce systematically different absolute cosines, so only the separation is
comparable across the three panels.

In [ ]:
def l2_normalize(Z):
    return Z / (np.linalg.norm(Z, axis=1, keepdims=True) + 1e-9)


# Helpers reused from U1-1_Embeddings-2_Images.ipynb
def make_fixed_pairs(y, n_pairs=20000, seed=0):
    rng = np.random.default_rng(seed)
    y = np.asarray(y)
    N = len(y)
    i = rng.integers(0, N, size=n_pairs)
    j = rng.integers(0, N, size=n_pairs)
    bad = (i == j)
    while bad.any():
        j[bad] = rng.integers(0, N, size=bad.sum())
        bad = (i == j)
    same = (y[i] == y[j])
    return i, j, same


def pairwise_cos_sims(Z, i, j):
    Z = l2_normalize(Z.astype(np.float32))
    return np.sum(Z[i] * Z[j], axis=1)


pair_i, pair_j, pair_same = make_fixed_pairs(y, n_pairs=5000, seed=RANDOM_STATE)

embeddings = {
    "MobileNetV2": Z_mobilenet,
    "EfficientNetV2M": Z_efficientnet,
    "CLIP (fastembed)": Z_clip,
}

fig, axes = plt.subplots(1, 3, figsize=(16, 3.5))
for ax, (name, Z) in zip(axes, embeddings.items()):
    sims = pairwise_cos_sims(Z, pair_i, pair_j)
    ax.hist(sims[pair_same], bins=30, alpha=0.65, label="same class")
    ax.hist(sims[~pair_same], bins=30, alpha=0.65, label="diff class")
    ax.set_title(name)
    ax.set_xlabel("cosine similarity")
    ax.legend()
axes[0].set_ylabel("count")
plt.tight_layout()
plt.show()

### 5.2 PCA + t-SNE projections

Two ways to look at a 1280-dimensional space on a flat screen, and they answer different questions.

**PCA** is a linear projection onto the two directions of greatest variance. Because it is linear,
distances in the plot mean something globally — but if the class structure is not aligned with the
top two components, PCA will not show it even when the space separates the classes perfectly.

**t-SNE** is nonlinear and optimizes to keep *near* points near. It usually produces much cleaner
visual clusters, at a price: cluster sizes and the distances *between* clusters are not meaningful,
and the layout changes with `perplexity` and the random seed. Never read a t-SNE plot as a map.

Treat both as illustrations, not evidence — section 5.3's linear probe is the quantitative version
of the same question.

In [ ]:
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

class_names = [name for name, _ in sorted(label_map.items(), key=lambda kv: kv[1])]

fig, axes = plt.subplots(2, 3, figsize=(16, 9))

for col, (name, Z) in enumerate(embeddings.items()):
    Zn = l2_normalize(Z)
    pca_2d = PCA(n_components=2, random_state=RANDOM_STATE).fit_transform(Zn)
    tsne_2d = TSNE(n_components=2, random_state=RANDOM_STATE, perplexity=30, init="pca").fit_transform(Zn)

    for row, proj, tag in [(0, pca_2d, "PCA"), (1, tsne_2d, "t-SNE")]:
        ax = axes[row, col]
        for cls_idx, cls_name in enumerate(class_names):
            mask = (y == cls_idx)
            ax.scatter(proj[mask, 0], proj[mask, 1], s=20, alpha=0.8, label=cls_name)
        ax.set_title(f"{name}\n{tag} (2D)")
        if row == 0 and col == 0:
            ax.legend()

plt.tight_layout()
plt.show()

### 5.3 Linear-probe accuracy

A single "quality number" per embedding: how well does a simple linear classifier separate cat vs.
dog in each space? The same stratified train/test split is used for all three, so the comparison is
apples-to-apples.

The *linear* part is the point. A sufficiently flexible model could tease the classes apart from
almost any representation, which would tell us about the classifier rather than the embedding. Fixing
it to logistic regression asks the question we actually care about: **has the frozen model already
arranged the space so that the classes are linearly separable?** That is the standard by which
embeddings are judged in the literature, and it is exactly what makes an embedding useful downstream
— you get to attach something small and cheap.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

idx_train, idx_test = train_test_split(
    np.arange(len(y)), test_size=0.25, stratify=y, random_state=RANDOM_STATE
)

probe_results = {}
for name, Z in embeddings.items():
    Zn = l2_normalize(Z)
    clf = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)
    clf.fit(Zn[idx_train], y[idx_train])
    acc = accuracy_score(y[idx_test], clf.predict(Zn[idx_test]))
    probe_results[name] = acc
    print(f"{name:22s} linear-probe test accuracy: {acc:.3f}")

plt.figure(figsize=(6, 4))
plt.bar(probe_results.keys(), probe_results.values())
plt.ylabel("test accuracy")
plt.ylim(0, 1)
plt.title("Linear-probe accuracy (cat vs. dog)")
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

### 5.4 Cosine similarity heatmap (curated subset)

The aggregate views above summarize thousands of pairs into distributions. This one zooms all the way
back in: four cats and four dogs, every pairwise cosine shown individually.

With the cats first and the dogs second, a well-behaved embedding shows a visible 2×2 block
structure — bright top-left and bottom-right blocks (within-class), darker off-diagonal blocks
(between-class). Individual bright spots in the off-diagonal blocks are worth looking at, since they
name the specific image pairs the space finds confusing.

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

rng2 = np.random.default_rng(RANDOM_STATE)
cat_idx = rng2.choice(np.where(y == label_map['cat'])[0], size=4, replace=False)
dog_idx = rng2.choice(np.where(y == label_map['dog'])[0], size=4, replace=False)
subset_idx = np.concatenate([cat_idx, dog_idx])
subset_labels = [labels[i] for i in subset_idx]

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for ax, (name, Z) in zip(axes, embeddings.items()):
    sim = cosine_similarity(l2_normalize(Z)[subset_idx])
    sns.heatmap(sim, ax=ax, square=True, vmin=0, vmax=1, cmap="nipy_spectral",
                xticklabels=subset_labels, yticklabels=subset_labels, cbar=False)
    ax.set_title(name)
plt.tight_layout()
plt.show()

## 6. CLIP's superpower: zero-shot text → image retrieval

Because CLIP embeds text and images in the same space, we can rank every image by its cosine
similarity to a **text prompt** -- no labels, no training. Neither MobileNetV2 nor EfficientNetV2M
(both vision-only) could do this, regardless of their size.

"Zero-shot" is the term for this: the model was never trained on cats-versus-dogs, never saw these
images, and was given no labeled examples — yet it can retrieve by an arbitrary description. Change
the query to something the label set does not contain at all (`"a black and white animal"`,
`"an animal on a couch"`) and it still ranks sensibly. That generality is what the contrastive
training signal bought.

One thing to expect in the numbers below: the cosines come out around **0.2-0.3**, not 0.9. That is
normal and is not a bug. CLIP has a well-documented *modality gap* -- image embeddings and text
embeddings occupy separate cones of the shared space, so a perfectly matched image/caption pair still
scores fairly low in absolute terms. Only the **ranking** is meaningful; never read a raw cross-modal
cosine as "confidence".

In [ ]:
from fastembed import TextEmbedding

text_prompts = ["a photo of a cat", "a photo of a dog"]

# Separate model from the vision tower, but the same 512-d space -- that shared space is
# the whole point of CLIP, and it is what makes the dot product below meaningful.
clip_text = TextEmbedding(model_name=CLIP_TEXT_MODEL_NAME)
Z_text = np.array(list(clip_text.embed(text_prompts)))
print("Z_text shape:", Z_text.shape)

query = "a photo of a cat"
query_vec = Z_text[text_prompts.index(query)]

sims_to_query = l2_normalize(Z_clip) @ l2_normalize(query_vec[None, :]).T
top_k = np.argsort(-sims_to_query.ravel())[:8]

fig, axes = plt.subplots(2, 4, figsize=(14, 7))
for ax, i in zip(axes.ravel(), top_k):
    ax.imshow(images[i])
    ax.set_title(f"{labels[i]}\ncos={sims_to_query[i, 0]:.3f}")
    ax.axis('off')
fig.suptitle(f'Top-8 images retrieved for text query: "{query}"')
plt.tight_layout()
plt.show()

## 7. Review

| | MobileNetV2 | EfficientNetV2M | CLIP (fastembed) |
|---|---|---|---|
| Training signal | Supervised ImageNet classification | Supervised ImageNet classification | Contrastive image-text matching |
| Architecture era / size | 2018, lightweight (~3.4M params) | 2021, large (~54M params) | 2021, ViT-B/32 (~151M params) |
| Modality | Vision only | Vision only | Vision **and** text, same space |
| Embedding dim | 1280 | 1280 | 512 |
| Runtime | Keras / TensorFlow | Keras / TensorFlow | ONNX Runtime (no torch) |
| Preprocessing | Rescale to [-1, 1] | Built into model (raw 0-255 in) | Handled inside `.embed()` |
| Output scale | Unnormalized | Unnormalized | L2-normalized by default |
| Text-query support | No | No | Yes |
| Typical use case | Fast/edge transfer learning | Higher-accuracy transfer learning | Zero-shot classification/retrieval, multimodal search |

**Takeaways**

- **Two frozen CNNs trained on the *same* ImageNet labels produce noticeably different embeddings**,
  purely from architecture and scale choices (lightweight MobileNetV2 vs. large EfficientNetV2M) --
  but both remain fundamentally vision-only. CLIP's different *training signal* (contrastive
  image-text matching) is what unlocks a genuinely new capability: querying images by text, which no
  ImageNet-supervised CNN can do, no matter how large.
- **The training signal determines what the space encodes.** An ImageNet-supervised model organizes
  images by the distinctions its 1,000 labels required. A contrastively-trained model organizes them
  by what captions people write. Neither is universally better — it depends on what you need the
  distances to mean.
- **Judge an embedding with a linear probe.** Cosine histograms and t-SNE plots are useful
  illustrations, but the probe answers the question that matters downstream: are the classes already
  linearly separable in this space?
- **Higher dimension is not automatically better.** CLIP's 512 dimensions carry more usable structure
  here than 1280 from either CNN. What matters is what the axes were trained to encode.
- **Absolute cosines are not comparable across models,** and cross-modal cosines are not confidences.
  Compare *separations* within a model, and read cross-modal scores only as a ranking.
- **On the plumbing:** swapping `transformers` for `fastembed` changed which runtime executes CLIP
  (ONNX Runtime instead of PyTorch), not which model runs -- both are OpenAI's CLIP ViT-B/32. Worth
  internalizing the general lesson: the *weights* and the *runtime that executes them* are separable
  choices. A model is not the framework you happen to load it with.

**Next:** `U2-2_CNN-8_Explainability.ipynb` stops asking what a model's features look like in
aggregate and starts asking, for a single prediction, which pixels actually drove it.